# 🧠 Tuần 3: Huấn luyện ViT-LoRA + Box Regression Head trên BraTS2020

**Tác giả:** Lâm Gia Bảo  
**Nhiệm vụ:** Huấn luyện ViT-LoRA + Box Regression Head trên tập lát cắt đa phương thức Dataset 4 (BraTS2020).  
Thực hiện thí nghiệm đối sánh hiệu quả phát hiện và đóng khung khối u giữa ảnh tổ hợp 3 kênh [FLAIR, T1ce, T2] so với 1 kênh T1ce đơn lẻ.

**Đặc điểm:**
- 1000 epochs tối đa + Early Stopping (patience=50)
- AdamW optimizer + CosineAnnealingWarmRestarts scheduler
- Lưu `loss_best.pth` và `loss_latest.pth` lên Google Drive
- Hỗ trợ resume từ checkpoint khi Colab hết thời gian

---
## ⚡ CÁCH DÙNG:
1. **Runtime → Change runtime type → T4 GPU**
2. **Chạy từ trên xuống dưới** (Ctrl+F9 = Run All)
3. Nếu Colab tắt giữa chừng → chạy lại từ đầu → tự resume

## Cell 0: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ======================================================================
# 🔧 CẤU HÌNH ĐƯỜNG DẪN - CHỈNH SỬA Ở ĐÂY NẾU CẦN
# ======================================================================

# Đường dẫn file zip BraTS2020_png trên Drive CÁ NHÂN
DRIVE_ZIP = '/content/drive/MyDrive/BrainTumorDetect/dataset/BraTS2020_png.zip'
# Nếu file zip nằm ở chỗ khác, sửa lại đường dẫn ở trên.
# Ví dụ: '/content/drive/MyDrive/datasets/BraTS2020_png.zip'

# Thư mục lưu checkpoint trên Drive (tự tạo)
DRIVE_CKPT = '/content/drive/MyDrive/BrainTumorDetect/checkpoints'

# Thư mục kết quả để gửi nhóm trưởng (tự tạo)
DRIVE_TEAM = '/content/drive/MyDrive/BrainTumorDetect/team_results'

# ======================================================================
os.makedirs(DRIVE_CKPT, exist_ok=True)
os.makedirs(DRIVE_TEAM, exist_ok=True)

# Kiểm tra file zip có tồn tại không
if os.path.exists(DRIVE_ZIP):
    size_gb = os.path.getsize(DRIVE_ZIP) / 1e9
    print(f'✅ Tìm thấy file zip: {DRIVE_ZIP} ({size_gb:.2f} GB)')
else:
    print(f'❌ KHÔNG tìm thấy file zip tại: {DRIVE_ZIP}')
    print('   Hãy sửa lại biến DRIVE_ZIP ở trên cho đúng đường dẫn!')
    # Thử tìm tự động
    import subprocess
    result = subprocess.run(['find', '/content/drive/MyDrive', '-name', 'BraTS2020_png.zip', '-maxdepth', '3'],
                          capture_output=True, text=True, timeout=30)
    if result.stdout.strip():
        print(f'   🔍 Tìm thấy tại: {result.stdout.strip()}')
        print(f'   Hãy copy đường dẫn này gán vào DRIVE_ZIP rồi chạy lại cell.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Tìm thấy file zip: /content/drive/MyDrive/BrainTumorDectect/dataset/BraTS2020_png.zip (3.67 GB)


## Cell 1: Clone Repo + Cài Dependencies + Giải nén Dataset

In [ ]:
import os, time
!pip uninstall -y torchao
# 1. Clone repo
%cd /content
if not os.path.exists('/content/BrainTumorDetect/.git'):
    !rm -rf BrainTumorDetect
    !git clone -b Lam_Gia_Bao_tuan3 https://github.com/MikanQuyt/BrainTumorDetect.git
else:
    print('Repo đã tồn tại, skip clone.')
%cd /content/BrainTumorDetect

# 2. Cài dependencies (gỡ torchao cũ của Colab gây lỗi với peft)
!pip uninstall -y torchao
!pip install -q peft transformers timm albumentations scipy scikit-learn opencv-python-headless nibabel h5py tqdm

# 3. Giải nén BraTS2020_png.zip vào dataset/
DEST = '/content/BrainTumorDetect/dataset/brats2020_png'
if os.path.isdir(DEST) and len(os.listdir(os.path.join(DEST, 'images', ''))) > 1000:
    print(f'✅ Dataset đã giải nén sẵn ({len(os.listdir(os.path.join(DEST, "images")))} ảnh)')
else:
    print(f'📦 Đang giải nén {DRIVE_ZIP} → {DEST} ...')
    t0 = time.time()
    # Giải nén vào thư mục tạm trước
    !mkdir -p /content/BrainTumorDetect/dataset
    # Thử xem cấu trúc bên trong zip
    import zipfile
    with zipfile.ZipFile(DRIVE_ZIP, 'r') as zf:
        # Lấy 1 file mẫu để xác định cấu trúc
        sample = zf.namelist()[0]
        print(f'   Cấu trúc zip: {sample}')

        # Nếu zip chứa BraTS2020_png/images/... → giải nén vào dataset/
        # Nếu zip chứa images/... trực tiếp → giải nén vào dataset/brats2020_png/
        if sample.startswith('BraTS2020_png/'):
            # Zip có thư mục gốc BraTS2020_png
            !cd /content/BrainTumorDetect/dataset && unzip -q -o "{DRIVE_ZIP}"
            # Rename nếu cần (BraTS2020_png → brats2020_png)
            if os.path.isdir('/content/BrainTumorDetect/dataset/BraTS2020_png') and not os.path.isdir(DEST):
                os.rename('/content/BrainTumorDetect/dataset/BraTS2020_png', DEST)
        elif sample.startswith('images/') or sample.startswith('masks/'):
            # Zip không có thư mục gốc
            os.makedirs(DEST, exist_ok=True)
            !cd "{DEST}" && unzip -q -o "{DRIVE_ZIP}"
        else:
            # Fallback: giải nén vào dataset/ và xem có gì
            !cd /content/BrainTumorDetect/dataset && unzip -q -o "{DRIVE_ZIP}"

    elapsed = time.time() - t0
    print(f'   ⏱️ Giải nén xong sau {elapsed:.0f}s')

# 4. Verify dataset
for subdir in ['images', 'masks']:
    p = os.path.join(DEST, subdir)
    if os.path.isdir(p):
        n = len([f for f in os.listdir(p) if f.endswith('.png')])
        print(f'   📂 {subdir}/: {n} files')
    else:
        print(f'   ❌ THIẾU: {p}')

print('\n✅ Setup hoàn tất!')

/content
Repo đã tồn tại, skip clone.
/content/BrainTumorDetect
📦 Đang giải nén /content/drive/MyDrive/BrainTumorDectect/dataset/BraTS2020_png.zip → /content/BrainTumorDetect/dataset/brats2020_png ...
   Cấu trúc zip: BraTS2020_png/
   ⏱️ Giải nén xong sau 85s
   📂 images/: 42034 files
   📂 masks/: 42034 files

✅ Setup hoàn tất!


## Cell 2: Verify Annotations + Fix đường dẫn

In [ ]:
import json, os
import pandas as pd

DEST = '/content/BrainTumorDetect/dataset/brats2020_png'
ANN_FILE = '/content/BrainTumorDetect/dataset/brats2020_annotations.json'
META_CSV = os.path.join(DEST, 'metadata.csv')

# Đọc trực tiếp từ metadata.csv có sẵn trong thư mục giải nén
if os.path.exists(META_CSV):
    print(f'📄 Đang tạo annotations từ {META_CSV} ...')
    df = pd.read_csv(META_CSV)
    records = []
    for _, row in df.iterrows():
        has_tumor = int(row['has_tumor'])
        gt_box = None
        if has_tumor and pd.notna(row.get('x_min')):
            gt_box = [int(row['x_min']), int(row['y_min']), int(row['x_max']) + 1, int(row['y_max']) + 1]

        png_p = str(row.get('png_path', f'images/{row["case_id"]}_z{int(row["z"]):03d}.png'))
        msk_p = str(row.get('mask_path', f'masks/{row["case_id"]}_z{int(row["z"]):03d}.png'))
        records.append({
            'dataset': 'brats2020',
            'case_id': str(row['case_id']),
            'slice': int(row['z']),
            'img_path': f'dataset/brats2020_png/{png_p}',
            'mask_path': f'dataset/brats2020_png/{msk_p}',
            'has_mask': True,
            'has_tumor': has_tumor,
            'sub_class_label': 1 if has_tumor else 0,
            'gt_box': gt_box,
            'split': str(row['split']),
            'fold': None
        })

    with open(ANN_FILE, 'w', encoding='utf-8') as f:
        json.dump(records, f, indent=2, ensure_ascii=False)

    n_total = len(records)
    n_train = sum(1 for r in records if r.get('split') == 'train')
    n_val = sum(1 for r in records if r.get('split') == 'val')
    n_test = sum(1 for r in records if r.get('split') == 'test')
    n_tumor = sum(1 for r in records if r.get('has_tumor') == 1)

    print(f'✅ Đã tạo thành công {n_total} records từ metadata.csv!')
    print(f'   Train: {n_train} | Val: {n_val} | Test: {n_test}')
    print(f'   Tumor: {n_tumor} | No Tumor: {n_total - n_tumor}')
else:
    print(f'❌ Không tìm thấy {META_CSV}')

📄 Đang tạo annotations từ /content/BrainTumorDetect/dataset/brats2020_png/metadata.csv ...
✅ Đã tạo thành công 42034 records từ metadata.csv!
   Train: 29772 | Val: 6087 | Test: 6175
   Tumor: 22814 | No Tumor: 19220


## Cell 3: Cấu hình + Khởi tạo Model

In [ ]:
import sys
sys.path.insert(0, '/content/BrainTumorDetect')

import torch
import numpy as np
import random
import json, csv, time, shutil
from pathlib import Path
from torch.utils.data import DataLoader

from data.unified_dataset import UnifiedBrainMRIDataset
from models.vit_lora import ViTLoRA
from utils.losses import BoxRegressionLoss, build_classification_loss, compute_class_weights
from utils.metrics import ClassificationMeter, BoxLocalizationMeter, detection_metrics, optimize_detection_threshold

# =================== CẤU HÌNH CHÍNH ===================
CFG = dict(
    # Model
    model_name='google/vit-base-patch16-224-in21k',
    lora_r=8, lora_alpha=16, lora_dropout=0.1,
    target_modules=['query', 'value'],
    use_box_head=True, sfp_dim=128, dropout=0.1,
    # Training
    max_epochs=1000, batch_size=32,
    lr_lora=3e-4, lr_head=1e-3, lr_box=1e-3,
    weight_decay=0.01,
    loss_type='focal', focal_gamma=2.0, label_smoothing=0.05,
    giou_weight=1.0, l1_weight=1.0, box_loss_weight=1.0,
    # Early Stopping
    patience=50, min_delta=1e-5,
    # Scheduler
    sched_T0=50, sched_Tmult=2, sched_eta_min=1e-6,
    # Detection
    threshold=0.35, box_padding=0.08,
    # Task
    task='multiclass', num_classes=4,
    datasets=['brats2020'],
    ann_files=['dataset/brats2020_annotations.json'],
    num_workers=2,
    # Misc
    seed=42, amp=True,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    save_every=10,
    ckpt_dir=DRIVE_CKPT,
)

CLASS_NAMES = {4: ['no_tumor', 'glioma', 'meningioma', 'pituitary'], 2: ['no_tumor', 'tumor']}

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_everything(CFG['seed'])

print(f"🖥️ Device: {CFG['device']}")
if CFG['device'] == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

🖥️ Device: cuda
   GPU: Tesla T4


AttributeError: 'torch._C._CudaDeviceProperties' object has no attribute 'total_mem'

## Cell 4: Build DataLoaders

In [ ]:
def build_ds(split, train=True):
    ds = UnifiedBrainMRIDataset(CFG['ann_files'], train=train, datasets=CFG['datasets'], split=split)
    rkey = 'sub_class_label' if CFG['task']=='multiclass' else 'has_tumor'
    valid = {0,1,2,3} if CFG['task']=='multiclass' else {0,1}
    ds.records = [r for r in ds.records if int(r[rkey]) in valid]
    return ds

train_ds = build_ds('train', True)
val_ds   = build_ds('val', False)
test_ds  = build_ds('test', False)
print(f'📊 train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}')

g = torch.Generator().manual_seed(CFG['seed'])
kw = dict(batch_size=CFG['batch_size'], num_workers=CFG['num_workers'],
          pin_memory=CFG['device'].startswith('cuda'), drop_last=False)
train_loader = DataLoader(train_ds, shuffle=True, generator=g, **kw)
val_loader   = DataLoader(val_ds, shuffle=False, **kw)
test_loader  = DataLoader(test_ds, shuffle=False, **kw)
print(f'📊 Batches: train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)}')

## Cell 5: Model + Loss + Optimizer + Scheduler

In [ ]:
# Class weights
rkey = 'sub_class_label' if CFG['task']=='multiclass' else 'has_tumor'
train_labels = [int(r[rkey]) for r in train_ds.records]
class_weights = compute_class_weights(train_labels, num_classes=CFG['num_classes'])

# Loss
cls_crit = build_classification_loss(CFG['loss_type'], class_weights, CFG['focal_gamma'], CFG['label_smoothing']).to(CFG['device'])
box_crit = BoxRegressionLoss(CFG['giou_weight'], CFG['l1_weight']).to(CFG['device'])

# Model
model = ViTLoRA(
    num_classes=CFG['num_classes'], lora_r=CFG['lora_r'], lora_alpha=CFG['lora_alpha'],
    lora_dropout=CFG['lora_dropout'], target_modules=CFG['target_modules'],
    use_box_head=CFG['use_box_head'], sfp_dim=CFG['sfp_dim'], dropout=CFG['dropout'],
    model_name=CFG['model_name'],
).to(CFG['device'])

pc = model.param_counts()
print(f'📐 Tham số: tổng={pc["total"]:,} | khả huấn={pc["trainable"]:,} ({pc["trainable_pct"]:.2f}%)')
print(f'   LoRA={pc["lora_params"]:,} ({pc["lora_pct_of_backbone"]:.3f}% backbone) | BoxHead={pc["box_head_params"]:,}')

# Optimizer
optimizer = torch.optim.AdamW(
    model.param_groups(lr_lora=CFG['lr_lora'], lr_head=CFG['lr_head'], lr_box=CFG['lr_box'], weight_decay=CFG['weight_decay'])
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=CFG['sched_T0'], T_mult=CFG['sched_Tmult'], eta_min=CFG['sched_eta_min']
)
scaler = torch.amp.GradScaler('cuda') if CFG['amp'] and CFG['device'].startswith('cuda') else None
print('✅ Model + Optimizer + Scheduler ready')

## Cell 6: Hàm Train/Eval + Checkpoint

In [ ]:
def lbl_key(task): return 'sub_class_label' if task=='multiclass' else 'binary_label'

def run_epoch(model, loader, cls_crit, box_crit, dev, task,
              blw=1.0, thr=0.35, opt=None, scaler=None, amp=False, desc=''):
    training = opt is not None
    model.train(training)
    nc = 4 if task=='multiclass' else 2
    cm = ClassificationMeter(CLASS_NAMES[nc])
    lm = BoxLocalizationMeter()
    tl, cl, bl, n = 0., 0., 0., 0
    ctx = torch.enable_grad() if training else torch.no_grad()
    ub = box_crit is not None and model.box_head is not None
    with ctx:
        for step, batch in enumerate(loader, 1):
            x = batch['image_224'].to(dev, non_blocking=True)
            y = batch[lbl_key(task)].to(dev, non_blocking=True)
            gb = batch['gt_box_norm'].to(dev, non_blocking=True)
            hb = batch['has_box'].to(dev, non_blocking=True)
            with torch.autocast(device_type=dev.split(':')[0], dtype=torch.float16, enabled=amp):
                out = model(x, return_dict=True)
                logits = out['logits']
                c_loss = cls_crit(logits.float(), y)
                if ub and out['pred_boxes'] is not None:
                    br = box_crit(out['pred_boxes'], gb, has_box=hb)
                    b_loss = br['loss']; loss = c_loss + blw * b_loss
                else:
                    b_loss = logits.sum()*0.0; loss = c_loss
            if training:
                opt.zero_grad(set_to_none=True)
                if scaler:
                    scaler.scale(loss).backward(); scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(opt); scaler.update()
                else:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt.step()
            bs = len(y)
            tl += float(loss.detach())*bs; cl += float(c_loss.detach())*bs; bl += float(b_loss.detach())*bs; n += bs
            cm.update(y.cpu().numpy(), logits.detach().float().softmax(1).cpu().numpy())
            if ub and out['pred_boxes'] is not None:
                pb = out['pred_boxes'].detach().cpu().numpy()
                gtb = gb.detach().cpu().numpy(); hbb = hb.detach().cpu().numpy()
                for i in range(bs):
                    if hbb[i]: lm.update(pred_box=pb[i], gt_box=gtb[i])
    m = cm.compute(with_detection=True)
    ya, pa = cm.arrays()
    m['detection_tau_tuned'] = detection_metrics(ya, pa, threshold=thr)
    m['loss'] = tl/max(n,1); m['cls_loss'] = cl/max(n,1); m['box_loss'] = bl/max(n,1)
    if lm.ious: m['box_head_localization'] = lm.summary()
    return m, cm

def save_ckpt(model, opt, sched, scaler, epoch, bvl, be, hist, pc, path, **kw):
    torch.save(dict(epoch=epoch, best_val_loss=bvl, best_epoch=be, patience_counter=pc,
        model_state_dict=model.trainable_state_dict(), optimizer_state_dict=opt.state_dict(),
        scheduler_state_dict=sched.state_dict(),
        scaler_state_dict=scaler.state_dict() if scaler else None,
        history=hist, config=CFG, **kw), path)

def save_model(model, path, epoch, vl, vm=None):
    torch.save(dict(
        state_dict=model.trainable_state_dict(), epoch=epoch, val_loss=vl,
        config=dict(num_classes=model.num_classes, lora_r=model.lora_r, lora_alpha=model.lora_alpha,
                    lora_dropout=model.lora_dropout, target_modules=list(model.target_modules),
                    use_box_head=model.use_box_head, input_norm=model.input_norm),
        val_metrics=vm), path)

def load_ckpt(model, opt, sched, scaler, path):
    ckpt = torch.load(path, map_location=CFG['device'], weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'], strict=False)
    opt.load_state_dict(ckpt['optimizer_state_dict'])
    sched.load_state_dict(ckpt['scheduler_state_dict'])
    if scaler and ckpt.get('scaler_state_dict'): scaler.load_state_dict(ckpt['scaler_state_dict'])
    return ckpt

print('✅ Functions defined')

## Cell 7: 🔥 TRAIN CHÍNH (1000 epochs + Early Stopping + Auto-Resume)

In [ ]:
resume_path = os.path.join(CFG['ckpt_dir'], 'training_checkpoint.pt')
best_path   = os.path.join(CFG['ckpt_dir'], 'loss_best.pth')
latest_path = os.path.join(CFG['ckpt_dir'], 'loss_latest.pth')

start_ep, best_vl, best_ep, pat_cnt, history = 1, float('inf'), 0, 0, []

if os.path.exists(resume_path):
    print('🔄 RESUME từ checkpoint...')
    ck = load_ckpt(model, optimizer, scheduler, scaler, resume_path)
    start_ep = ck['epoch'] + 1
    best_vl, best_ep, pat_cnt = ck['best_val_loss'], ck['best_epoch'], ck['patience_counter']
    history = ck.get('history', [])
    print(f'   → Resumed from epoch {ck["epoch"]}, best_vl={best_vl:.6f} (ep {best_ep}), patience={pat_cnt}/{CFG["patience"]}')
else:
    print('🆕 Bắt đầu huấn luyện từ đầu')

print(f'\n{"="*70}')
print(f'  ViT-LoRA (r=8) + BoxHead | BraTS2020 | Max {CFG["max_epochs"]} eps | ES patience={CFG["patience"]}')
print(f'{"="*70}\n')

t0 = time.time()
for epoch in range(start_ep, CFG['max_epochs'] + 1):
    et0 = time.time()
    tr, _ = run_epoch(model, train_loader, cls_crit, box_crit, CFG['device'], CFG['task'],
                      blw=CFG['box_loss_weight'], thr=CFG['threshold'],
                      opt=optimizer, scaler=scaler, amp=CFG['amp'])
    va, _ = run_epoch(model, val_loader, cls_crit, box_crit, CFG['device'], CFG['task'],
                      blw=CFG['box_loss_weight'], thr=CFG['threshold'], amp=CFG['amp'])
    scheduler.step(epoch)
    lrs = [pg['lr'] for pg in optimizer.param_groups]
    vbi = va.get('box_head_localization', {}).get('mean_box_iou', float('nan'))
    et = time.time() - et0

    history.append(dict(epoch=epoch, train_loss=tr['loss'], train_cls=tr['cls_loss'], train_box=tr['box_loss'],
        val_loss=va['loss'], val_cls=va['cls_loss'], val_box=va['box_loss'],
        val_acc=va['accuracy'], val_f1=va['f1'], val_recall=va['recall'],
        val_box_iou=vbi, lr=lrs[0], secs=round(et,1)))

    vl = va['loss']
    improved = vl < (best_vl - CFG['min_delta'])
    if improved:
        best_vl, best_ep, pat_cnt = vl, epoch, 0
        save_model(model, best_path, epoch, vl, va)
        mk = ' ⭐ BEST'
    else:
        pat_cnt += 1
        mk = f' (pat {pat_cnt}/{CFG["patience"]})'
    save_model(model, latest_path, epoch, vl, va)

    if epoch <= 5 or epoch % 5 == 0 or improved or pat_cnt >= CFG['patience']:
        print(f'E{epoch:4d} | loss: {tr["loss"]:.5f}/{vl:.5f} | f1={va["f1"]:.4f} rec={va["recall"]:.4f} biou={vbi:.4f} | lr={lrs[0]:.2e} | {et:.0f}s{mk}')

    if epoch % CFG['save_every'] == 0:
        save_ckpt(model, optimizer, scheduler, scaler, epoch, best_vl, best_ep, history, pat_cnt, resume_path)

    if pat_cnt >= CFG['patience']:
        print(f'\n🛑 EARLY STOPPING @ epoch {epoch} (best={best_ep}, val_loss={best_vl:.6f})')
        save_ckpt(model, optimizer, scheduler, scaler, epoch, best_vl, best_ep, history, pat_cnt, resume_path)
        break

tt = time.time() - t0
print(f'\n✅ Done! {tt/3600:.2f}h | Best val_loss={best_vl:.6f} @ epoch {best_ep}')
print(f'   📂 {best_path}')
print(f'   📂 {latest_path}')

## Cell 8: Lưu History + Vẽ Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Save CSV
df = pd.DataFrame(history)
df.to_csv(os.path.join(CFG['ckpt_dir'], 'history_3ch.csv'), index=False)

# Plot
if len(history) > 1:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    ep = df['epoch']
    axes[0,0].plot(ep, df['train_loss'], label='Train'); axes[0,0].plot(ep, df['val_loss'], label='Val')
    axes[0,0].axvline(best_ep, color='r', ls='--', alpha=.5); axes[0,0].set_title('Total Loss'); axes[0,0].legend(); axes[0,0].set_yscale('log')
    axes[0,1].plot(ep, df['train_cls'], label='Train Cls'); axes[0,1].plot(ep, df['val_cls'], label='Val Cls')
    axes[0,1].set_title('Classification Loss'); axes[0,1].legend()
    axes[0,2].plot(ep, df['train_box'], label='Train Box'); axes[0,2].plot(ep, df['val_box'], label='Val Box')
    axes[0,2].set_title('Box Loss (GIoU+L1)'); axes[0,2].legend()
    axes[1,0].plot(ep, df['val_acc'], label='Acc'); axes[1,0].plot(ep, df['val_f1'], label='F1')
    axes[1,0].plot(ep, df['val_recall'], label='Recall'); axes[1,0].set_title('Val Metrics'); axes[1,0].legend()
    vi = df.dropna(subset=['val_box_iou'])
    if len(vi)>0: axes[1,1].plot(vi['epoch'], vi['val_box_iou'], color='green')
    axes[1,1].set_title('Val Box IoU')
    axes[1,2].plot(ep, df['lr']); axes[1,2].set_title('Learning Rate'); axes[1,2].set_yscale('log')
    plt.suptitle(f'3-ch [FLAIR,T1ce,T2] | Best loss={best_vl:.5f} @ ep {best_ep}', fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(CFG['ckpt_dir'], 'curves_3ch.png'), dpi=150, bbox_inches='tight')
    plt.show()
print('✅ History saved')

## Cell 9: Đánh giá Test Set (3 kênh)

In [ ]:
# Load best
bk = torch.load(best_path, map_location=CFG['device'], weights_only=False)
model.load_state_dict(bk['state_dict'], strict=False)
print(f'Loaded best model (epoch {bk["epoch"]}, val_loss={bk["val_loss"]:.6f})')

tm, tc = run_epoch(model, test_loader, cls_crit, box_crit, CFG['device'], CFG['task'],
                   blw=CFG['box_loss_weight'], thr=CFG['threshold'], amp=CFG['amp'])

ya, pa = tc.arrays()
tm['threshold_opt'] = optimize_detection_threshold(ya, pa, target_recall=0.95)['best']

print(f'\n{"="*60}')
print(f'📊 TEST — 3 kênh [FLAIR, T1ce, T2]')
print(f'{"="*60}')
print(f'  Loss: {tm["loss"]:.5f} (cls={tm["cls_loss"]:.5f} box={tm["box_loss"]:.5f})')
print(f'  Acc={tm["accuracy"]:.4f}  Prec={tm["precision"]:.4f}  Rec={tm["recall"]:.4f}  F1={tm["f1"]:.4f}')
if 'box_head_localization' in tm:
    l=tm['box_head_localization']
    print(f'  BoxIoU={l["mean_box_iou"]:.4f}  IoU@50={l["iou_at_50"]:.4f}  IoU@75={l["iou_at_75"]:.4f}')
d=tm['detection_tau_tuned']
print(f'  Detection(τ={CFG["threshold"]}): rec={d["recall"]:.4f} spec={d["specificity"]:.4f} f1={d["f1"]:.4f}')

with open(os.path.join(CFG['ckpt_dir'], 'test_3ch.json'), 'w') as f:
    json.dump(tm, f, indent=2, default=str)
test_3ch = tm  # lưu lại để so sánh

## Cell 10: 🔬 ĐỐI SÁNH — Train T1ce-only

In [ ]:
class T1ceOnly(torch.utils.data.Dataset):
    """Chỉ dùng kênh T1ce (index 1), nhân 3 cho tương thích ViT."""
    def __init__(self, base): self.base = base
    def __len__(self): return len(self.base)
    def __getitem__(self, i):
        s = self.base[i]
        s['image_224'] = s['image_224'][1:2].expand(3,-1,-1).contiguous()
        s['image_1024'] = s['image_1024'][1:2].expand(3,-1,-1).contiguous()
        return s

tl_t = DataLoader(T1ceOnly(build_ds('train',True)), shuffle=True, generator=torch.Generator().manual_seed(42), **kw)
vl_t = DataLoader(T1ceOnly(build_ds('val',False)), shuffle=False, **kw)
tsl_t = DataLoader(T1ceOnly(build_ds('test',False)), shuffle=False, **kw)

# Fresh model
m2 = ViTLoRA(num_classes=CFG['num_classes'], lora_r=CFG['lora_r'], lora_alpha=CFG['lora_alpha'],
    lora_dropout=CFG['lora_dropout'], target_modules=CFG['target_modules'],
    use_box_head=True, sfp_dim=CFG['sfp_dim'], dropout=CFG['dropout'], model_name=CFG['model_name']).to(CFG['device'])
cc2 = build_classification_loss(CFG['loss_type'], class_weights, CFG['focal_gamma'], CFG['label_smoothing']).to(CFG['device'])
bc2 = BoxRegressionLoss(CFG['giou_weight'], CFG['l1_weight']).to(CFG['device'])
op2 = torch.optim.AdamW(m2.param_groups(lr_lora=CFG['lr_lora'], lr_head=CFG['lr_head'], lr_box=CFG['lr_box'], weight_decay=CFG['weight_decay']))
sc2 = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(op2, T_0=CFG['sched_T0'], T_mult=CFG['sched_Tmult'], eta_min=CFG['sched_eta_min'])
amp2 = torch.amp.GradScaler('cuda') if CFG['amp'] and CFG['device'].startswith('cuda') else None

T1_MAX = 200  # Ngắn hơn cho thí nghiệm đối sánh
res_t1 = os.path.join(CFG['ckpt_dir'], 'ckpt_t1ce.pt')
best_t1 = os.path.join(CFG['ckpt_dir'], 'loss_best_t1ce.pth')
se, bv, be2, pc2, h2 = 1, float('inf'), 0, 0, []
if os.path.exists(res_t1):
    ck = load_ckpt(m2, op2, sc2, amp2, res_t1)
    se, bv, be2, pc2, h2 = ck['epoch']+1, ck['best_val_loss'], ck['best_epoch'], ck['patience_counter'], ck.get('history',[])
    print(f'🔄 T1ce resumed from ep {ck["epoch"]}')

print(f'\n🔬 Training T1ce-only (max {T1_MAX} epochs)...')
t1_t0 = time.time()
for ep in range(se, T1_MAX+1):
    tr2,_ = run_epoch(m2, tl_t, cc2, bc2, CFG['device'], CFG['task'], blw=1.0, thr=0.35, opt=op2, scaler=amp2, amp=CFG['amp'])
    va2,_ = run_epoch(m2, vl_t, cc2, bc2, CFG['device'], CFG['task'], blw=1.0, thr=0.35, amp=CFG['amp'])
    sc2.step(ep)
    bi2 = va2.get('box_head_localization',{}).get('mean_box_iou',float('nan'))
    h2.append(dict(epoch=ep, train_loss=tr2['loss'], val_loss=va2['loss'], val_f1=va2['f1'], val_recall=va2['recall'], val_box_iou=bi2))
    if va2['loss'] < (bv - 1e-5):
        bv, be2, pc2 = va2['loss'], ep, 0
        save_model(m2, best_t1, ep, va2['loss'], va2)
        if ep%10==0: print(f'  T1ce E{ep}: vl={va2["loss"]:.5f} f1={va2["f1"]:.4f} biou={bi2:.4f} ⭐')
    else: pc2 += 1
    if ep % 10 == 0: save_ckpt(m2, op2, sc2, amp2, ep, bv, be2, h2, pc2, res_t1)
    if pc2 >= CFG['patience']:
        print(f'🛑 T1ce Early Stop @ ep {ep} (best={be2})')
        save_ckpt(m2, op2, sc2, amp2, ep, bv, be2, h2, pc2, res_t1)
        break
print(f'✅ T1ce done: {(time.time()-t1_t0)/60:.1f}min, best_vl={bv:.6f} @ ep {be2}')

## Cell 11: Đánh giá T1ce + Bảng so sánh

In [ ]:
bk2 = torch.load(best_t1, map_location=CFG['device'], weights_only=False)
m2.load_state_dict(bk2['state_dict'], strict=False)
tm2,_ = run_epoch(m2, tsl_t, cc2, bc2, CFG['device'], CFG['task'], blw=1.0, thr=0.35, amp=CFG['amp'])

print(f'\n{"="*60}')
print(f'📊 TEST — 1 kênh T1ce')
print(f'{"="*60}')
print(f'  Loss={tm2["loss"]:.5f} Acc={tm2["accuracy"]:.4f} F1={tm2["f1"]:.4f} Rec={tm2["recall"]:.4f}')
if 'box_head_localization' in tm2:
    l2=tm2['box_head_localization']; print(f'  BoxIoU={l2["mean_box_iou"]:.4f}')

# BẢNG SO SÁNH
def row(m, name):
    loc = m.get('box_head_localization', {})
    return dict(Config=name, Loss=f'{m["loss"]:.5f}', Acc=f'{m["accuracy"]:.4f}',
        Prec=f'{m["precision"]:.4f}', Recall=f'{m["recall"]:.4f}', F1=f'{m["f1"]:.4f}',
        BoxIoU=f'{loc.get("mean_box_iou",float("nan")):.4f}')

comp = pd.DataFrame([row(test_3ch, '3-ch [FLAIR,T1ce,T2]'), row(tm2, '1-ch T1ce only')])
print(f'\n{"="*70}')
print('📊 ĐỐI SÁNH: 3 Kênh vs 1 Kênh T1ce')
print('='*70)
print(comp.set_index('Config').T.to_string())

comp.to_csv(os.path.join(CFG['ckpt_dir'], 'comparison.csv'), index=False)
with open(os.path.join(CFG['ckpt_dir'], 'comparison.json'), 'w') as f:
    json.dump({'3ch': test_3ch, 't1ce': tm2}, f, indent=2, default=str)

# Bar chart
fig, ax = plt.subplots(figsize=(8,5))
names = ['Accuracy','Recall','F1']
v3 = [test_3ch['accuracy'], test_3ch['recall'], test_3ch['f1']]
v1 = [tm2['accuracy'], tm2['recall'], tm2['f1']]
if 'box_head_localization' in test_3ch:
    names.append('Box IoU')
    v3.append(test_3ch['box_head_localization']['mean_box_iou'])
    v1.append(tm2.get('box_head_localization',{}).get('mean_box_iou',0))
x = np.arange(len(names)); w=0.35
ax.bar(x-w/2, v3, w, label='3-ch [FLAIR,T1ce,T2]', color='steelblue')
ax.bar(x+w/2, v1, w, label='1-ch T1ce', color='coral')
ax.set_xticks(x); ax.set_xticklabels(names); ax.legend(); ax.set_ylim(0,1.05)
ax.set_title('3-ch vs T1ce on BraTS2020 Test Set', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(CFG['ckpt_dir'], 'comparison.png'), dpi=150)
plt.show()

## Cell 12: 🚀 COPY KẾT QUẢ LÊN DRIVE NHÓM

In [ ]:
MY_DIR = os.path.join(DRIVE_TEAM, 'LamGiaBao_Week3')
os.makedirs(MY_DIR, exist_ok=True)

files = [
    (best_path, 'loss_best.pth'),
    (latest_path, 'loss_latest.pth'),
    (os.path.join(CFG['ckpt_dir'], 'history_3ch.csv'), 'history_3ch.csv'),
    (os.path.join(CFG['ckpt_dir'], 'curves_3ch.png'), 'curves_3ch.png'),
    (os.path.join(CFG['ckpt_dir'], 'test_3ch.json'), 'test_3ch.json'),
    (os.path.join(CFG['ckpt_dir'], 'comparison.csv'), 'comparison.csv'),
    (os.path.join(CFG['ckpt_dir'], 'comparison.json'), 'comparison.json'),
    (os.path.join(CFG['ckpt_dir'], 'comparison.png'), 'comparison.png'),
]
print(f'📂 Copying to {MY_DIR}:')
for src, name in files:
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(MY_DIR, name))
        print(f'  ✅ {name} ({os.path.getsize(src)/1e6:.1f} MB)')
    else:
        print(f'  ⚠️ {name} (not found)')

print(f'\n🎉 HOÀN TẤT! Files tại: {MY_DIR}')
print(f'   Copy thư mục LamGiaBao_Week3 vào link Drive nhóm:')
print(f'   https://drive.google.com/drive/folders/1JUGAxnRcgTf4INQstlUt9YqggO_cbWZ6')